In [1]:
import os 
import glob
import pandas as pd

In [2]:
model_suffix = "g-n"
projections_path = "../output/projections"
output_path = "../output/combined_projections.csv"

In [3]:
rows_list = []

In [4]:
csv_files = glob.glob(os.path.join(projections_path, "*_projections.csv"))

for file_path in csv_files:
    filename = os.path.basename(file_path)

    base_name = filename.replace("_projections.csv", "")

    if model_suffix not in base_name:
        continue

    prefix, _ = base_name.rsplit(f"-{model_suffix}", 1)
    model_variation = prefix.split("-")[-1]
    model_name = "-".join(prefix.split("-")[:-1])
    model_id = f"{model_name}-{model_variation}-{model_suffix}"

    try:
        model_data = pd.read_csv(file_path)
    except Exception:
        continue

    dimensions = model_data["Dimension"].unique()

    for dimension in dimensions:
        if dimension in ["Warmth", "Competence"]:
            continue

        mask = (
            (model_data["Model"] == model_id) &
            (model_data["Dimension"] == dimension)
        )
        filtered_data = model_data[mask]

        f_mean = filtered_data["Female Names_mean"].iloc[0]
        m_mean = filtered_data["Male Names_mean"].iloc[0]

        rows_list.append({
            "model_name": model_name,
            "model_variation": f"{model_variation}-{model_suffix}",
            "dimension": dimension,
            "female_names": f_mean,
            "male_names": m_mean
        })

result_df = pd.DataFrame(rows_list)

In [5]:
result_df

,model_name,model_variation,dimension,female_names,male_names
0,Qwen3-8B,democratic_9-g-n,Sociability,0.412652,-0.412652
1,Qwen3-8B,democratic_9-g-n,Morality,0.491826,-0.491826
2,Qwen3-8B,democratic_9-g-n,Ability,-0.062953,0.062953
3,Qwen3-8B,democratic_9-g-n,Agency,-0.440775,0.440775
4,Qwen3-8B,democratic_9-g-n,Status,-0.076714,0.076714
...,...,...,...,...,...
646,Apertus-8B-Instruct-2509,liberal_5-g-n,Ability,-0.034514,0.034514
647,Apertus-8B-Instruct-2509,liberal_5-g-n,Agency,-0.437885,0.437885
648,Apertus-8B-Instruct-2509,liberal_5-g-n,Status,-0.170846,0.170846
649,Apertus-8B-Instruct-2509,liberal_5-g-n,Politics,0.159804,-0.159804


In [6]:
result_df.to_csv(output_path)